# Lesson 2

This lesson will teach me how to remember past prompts (enables you to create a chatbot that remembers past conversations)

In [14]:
# start by importing openai key and tools 

import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')


from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory


In [15]:
# Set LLM model
llm_model = "gpt-3.5-turbo"

# Build LLM: this builds an LLM that you can have a conversation with where it remembers past prompts and responses
llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True # Setting this as true means that the LLM will show you what it is doing, setting it to False will mean it only shows the responses/outputs, not how it got the output
)


In [16]:
# starting the conversation with the first prompt
conversation.invoke({"input": "Hi, my name is Andrew"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Andrew
AI:

> Finished chain.


{'input': 'Hi, my name is Andrew',
 'history': '',
 'response': "Hello Andrew! It's nice to meet you. How can I assist you today?"}

In [17]:
conversation.invoke({"input": "What is 1+1?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! It's nice to meet you. How can I assist you today?
Human: What is 1+1?
AI:

> Finished chain.


{'input': 'What is 1+1?',
 'history': "Human: Hi, my name is Andrew\nAI: Hello Andrew! It's nice to meet you. How can I assist you today?",
 'response': '1+1 equals 2. Is there anything else you would like to know?'}

In [18]:
conversation.invoke({"input": "What is my name?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! It's nice to meet you. How can I assist you today?
Human: What is 1+1?
AI: 1+1 equals 2. Is there anything else you would like to know?
Human: What is my name?
AI:

> Finished chain.


{'input': 'What is my name?',
 'history': "Human: Hi, my name is Andrew\nAI: Hello Andrew! It's nice to meet you. How can I assist you today?\nHuman: What is 1+1?\nAI: 1+1 equals 2. Is there anything else you would like to know?",
 'response': 'Your name is Andrew. Is there anything else you would like to know or discuss?'}

In [19]:
print(memory.buffer)



Human: Hi, my name is Andrew
AI: Hello Andrew! It's nice to meet you. How can I assist you today?
Human: What is 1+1?
AI: 1+1 equals 2. Is there anything else you would like to know?
Human: What is my name?
AI: Your name is Andrew. Is there anything else you would like to know or discuss?


In [20]:
memory.load_memory_variables({})


{'history': "Human: Hi, my name is Andrew\nAI: Hello Andrew! It's nice to meet you. How can I assist you today?\nHuman: What is 1+1?\nAI: 1+1 equals 2. Is there anything else you would like to know?\nHuman: What is my name?\nAI: Your name is Andrew. Is there anything else you would like to know or discuss?"}

In [21]:
# LangChain stores the conversation with ConversationBufferMemory()
memory = ConversationBufferMemory()
print(memory)

chat_memory=InMemoryChatMessageHistory(messages=[])


In [22]:
# you can explicitly add things to conversation memory in the following way
memory.save_context({"input": "Hi"}, 
                    {"output": "What's up"})

# show memory
print(memory.buffer)

Human: Hi
AI: What's up


In [23]:
# you can continue saving data to the memory with the same command
memory.save_context({"input": "Not much, just hanging"}, 
                    {"output": "Cool"})

# show memory
print(memory.buffer)


Human: Hi
AI: What's up
Human: Not much, just hanging
AI: Cool


In [24]:
memory.load_memory_variables({})

{'history': "Human: Hi\nAI: What's up\nHuman: Not much, just hanging\nAI: Cool"}

LLM's are 'stateless' (they do not remember the past interactions), meaning that each transaction is independent. Chatbots appear to have memory by providing the full conversation as 'context'. As the conversation increases in length, the amount of memory required to store the context that is provided to the LLM can become really big. So LangChain provides several kinds of 'memory' to store and accumulate conversation.

# Limiting the number of conversational exchanges saved: ConversationBufferWindowMemory

Using this method caps the memory needed as conversations become incredibly long, as only a certain number of (the most recent) conversational exchanges are stored

In [34]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=1) # k=1 means that we want the memory to only store one conversational exchange

In [35]:
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
# you can see here that it only remembers the most recent interactions/conversation exchange
memory.load_memory_variables({})

{'history': 'Human: Not much, just hanging\nAI: Cool'}

In [45]:
# Build an LLM with only a window of memory, not all of the conversation exchanges

llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=False
)

In [46]:
conversation.predict(input="Hi, my name is Andrew")


"Hello Andrew! It's nice to meet you. How can I assist you today?"

In [47]:
conversation.predict(input="What is 1+1?")


'1+1 equals 2. Is there anything else you would like to know?'

In [48]:
# you can see here that it doesn't remember Andrew's name because only the most previous conversation exchange is stored in memory
conversation.predict(input="What is my name?")

"I'm sorry, I do not have access to personal information such as your name. Is there anything else you would like to know?"

# Limiting the number of tokens saved: ConversationTokenBufferMemory

A lot of LLM's pricing is based on number of tokens so this maps more directly to the cost of the LLM calls

In [52]:
# imports
from langchain.memory import ConversationTokenBufferMemory
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI


In [53]:
# building llm
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [ ]:
# setting the type of memory and setting the max number of tokens saved
# You have to define the llm being used because different types of llms use different ways of counting tokens
# changing the token limit changes the number of tokens saved
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=30) 

# saving some pre-existing context (Using statements begining with A-B-C-etc to track when things were said)
memory.save_context({"input": "AI is what?!"},
                    {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"},
                    {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, 
                    {"output": "Charming!"})

In [60]:
memory.load_memory_variables({})

{'history': 'AI: Beautiful!\nHuman: Chatbots are what?\nAI: Charming!'}

# Limiting the memory by getting the LLM to write a summary of the conversation so far: ConversationSummaryMemory

In [64]:
from langchain.memory import ConversationSummaryBufferMemory

# create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

# create memory
# with a high token limit eg 400 you could store all the text, but with a lower token limit it would create a summary of the text
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100) 

# insert into memory a few conversational terms
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"}, 
                    {"output": f"{schedule}"})

In [63]:
memory.load_memory_variables({})

{'history': 'System: The human and AI exchange greetings and discuss the schedule for the day, including a meeting with the product team, work on the LangChain project, and a lunch meeting with a customer interested in AI. The AI provides details on each event and emphasizes the power of LangChain as a tool.'}

In [65]:
# create a conversation chain to use the LLM
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

In [71]:
conversation.predict(input="What would be a good demo to show?")
# In the current conversation it shows under the system section a summary of the conversation so far



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human and AI discuss the schedule for the day, including a meeting with the product team, work on the LangChain project, and a lunch meeting with a customer interested in AI. The AI provides details on each event and emphasizes the power of LangChain as a tool. For the meeting with the product team, a demo showcasing the latest features and improvements in LangChain would be ideal, including a live demonstration of how it streamlines language translation processes and improves efficiency. Highlighting recent success stories or case studies would also be beneficial to showcase its real-world impact. For the work on the LangChain project, a demo focusing on t

'A good demo to show for the LangChain project could be a side-by-side comparison of traditional language translation methods versus using LangChain. This could highlight the speed, accuracy, and cost-effectiveness of using LangChain for language translation tasks. Additionally, showcasing a live demonstration of how LangChain can handle multiple languages simultaneously or adapt to different dialects and nuances could impress the product team and demonstrate the versatility of the platform.'

In [72]:
memory.load_memory_variables({})

{'history': "System: The human and AI discuss the schedule for the day, including a meeting with the product team, work on the LangChain project, and a lunch meeting with a customer interested in AI. The AI provides details on each event and emphasizes the power of LangChain as a tool. For the meeting with the product team, a demo showcasing the latest features and improvements in LangChain would be ideal, including a live demonstration of how it streamlines language translation processes and improves efficiency. Highlighting recent success stories or case studies would also be beneficial to showcase its real-world impact. For the work on the LangChain project, a demo focusing on the technical aspects of the platform would be great. This could involve showcasing the architecture of LangChain, the algorithms used for language translation, and any recent updates or enhancements made to the system. Demonstrating how LangChain integrates with other AI technologies or platforms could also b

# Additional memory types

Vector data memory
* Stores test (from conversation or elsewhere) in a vector database and retrieves the most relevant blocks of text

Entity memories
* Using an LLM, it remembers details about specific entities

You can use multiple memories at one time eg conversation memory and entity memory to recall individuals.

You can also store the conversation in a conventional database (such as key-value store or SQL).

